# 02. Feature Engineering

EDA 결과로 고른 feature와 세트(A/B/C)를 만들고, **그룹별 상관**으로 확인한다(통합 상관은 Batch 효과가 섞인다).

In [ ]:
import sys; sys.path.append('../src')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from features import load_cells, model_cells, FEATURE_SETS
from matplotlib import font_manager

fonts = {f.name for f in font_manager.fontManager.ttflist}      # 설치된 한글 폰트를 고른다

plt.rcParams.update({'font.family': next((f for f in ['AppleGothic', 'NanumGothic', 'Malgun Gothic'] if f in fonts), 'DejaVu Sans'),
                     'axes.unicode_minus': False})
df = model_cells()
df['g'] = df.batch + np.where(df.newstruct, '-new', '-legacy')
for k, v in FEATURE_SETS.items():
    print(k, v)

df[['uid', 'g', 'cycle_life'] + FEATURE_SETS['C']].head()

A ['dq_logvar']
B ['dq_logvar', 'contrast', 'avg_c', 'soc1']
C ['dq_logvar', 'contrast', 'avg_c', 'soc1', 'tavg_mean']


,uid,g,cycle_life,dq_logvar,contrast,avg_c,soc1,tavg_mean
0,b1_c05,b1-legacy,1074.0,-4.178878,0.0,4.400000,80.0,30.921498
1,b1_c06,b1-legacy,636.0,-3.768878,0.0,4.800000,80.0,32.273845
2,b1_c07,b1-legacy,870.0,-3.813486,0.0,4.800000,80.0,31.990085
3,b1_c09,b1-legacy,1054.0,-4.059384,1.8,4.320000,40.0,32.641534
4,b1_c11,b1-legacy,788.0,-4.146897,2.4,4.153846,50.0,32.759323


## 그룹별 Spearman 상관 (vs log10 cycle_life)

In [ ]:
feats = FEATURE_SETS['C']
rows = {'통합': [df[f].corr(df.y, method='spearman') for f in feats]}

for g, d in df.groupby('g'):
    if len(d) >= 20: rows[g] = [d[f].corr(d.y, method='spearman') for f in feats]

pd.DataFrame(rows, index=feats).round(2)

,통합,b1-legacy,b2-legacy,b3-new
dq_logvar,-0.89,-0.83,-0.38,-0.80
contrast,-0.11,-0.22,-0.58,-0.53
avg_c,-0.04,-0.43,0.16,-0.13
soc1,0.08,-0.06,0.40,0.44
tavg_mean,0.33,-0.12,-0.20,0.03


## feature 간 상관 (중복 확인)

In [3]:
df[feats].corr(method='spearman').round(2)

,dq_logvar,contrast,avg_c,soc1,tavg_mean
dq_logvar,1.00,0.24,-0.04,-0.09,-0.37
contrast,0.24,1.00,-0.44,-0.41,0.03
avg_c,-0.04,-0.44,1.00,0.08,0.14
soc1,-0.09,-0.41,0.08,1.00,-0.27
tavg_mean,-0.37,0.03,0.14,-0.27,1.00


In [5]:
df[feats + ['cycle_life']].corr(method='spearman').round(2)

,dq_logvar,contrast,avg_c,soc1,tavg_mean,cycle_life
dq_logvar,1.00,0.24,-0.04,-0.09,-0.37,-0.89
contrast,0.24,1.00,-0.44,-0.41,0.03,-0.11
avg_c,-0.04,-0.44,1.00,0.08,0.14,-0.04
soc1,-0.09,-0.41,0.08,1.00,-0.27,0.08
tavg_mean,-0.37,0.03,0.14,-0.27,1.00,0.33
cycle_life,-0.89,-0.11,-0.04,0.08,0.33,1.00


## 선택 근거

- `dq_logvar`: 그룹 안에서도 일관되게 강하다(b1 −0.84, b3 −0.76).
- `contrast`, `avg_c`, `soc1`: 충전 정책. 사이클 시작 전부터 알 수 있지만 효과 방향이 Batch마다 달라서 보조로 두고 세트 B에서 기여를 확인한다.
- `tavg_mean`: 온도. 세트 C에서 추가 기여를 확인한다.
- 제외: 초기 QD·IR(Batch 지문), knee·후반 열화 속도(수명 후반 정보라 누수).